# Ollama on Google Colab with GPU

This notebook sets up Ollama with Llama 3 on Google Colab's free GPU and exposes it via ngrok.

**Steps:**
1. Run all cells in order
2. Copy the ngrok URL (will look like: `https://xxxx-xx-xxx-xxx-xx.ngrok-free.app`)
3. Update your local `.env` file with: `LLM_BASE_URL=<your-ngrok-url>`
4. Keep this notebook running while using your RAG system

**Note:** Free Colab sessions timeout after ~12 hours of inactivity or ~24 hours total.

In [ ]:
# Check GPU availability
!nvidia-smi

In [ ]:
# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# Install pyngrok for tunneling
!pip install pyngrok -q

In [ ]:
# Start Ollama server in background
import subprocess
import time

# Start Ollama server
ollama_process = subprocess.Popen(['ollama', 'serve'], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
print("Starting Ollama server...")
time.sleep(5)
print("✓ Ollama server started")

In [ ]:
# Pull Llama 3 model (this will take a few minutes)
!ollama pull llama3
print("\n✓ Llama 3 model downloaded")

In [ ]:
# Test Ollama locally
!curl http://localhost:11434/api/generate -d '{
  "model": "llama3",
  "prompt": "What is 2+2?",
  "stream": false
}'

In [ ]:
# Setup ngrok tunnel
# Get your free ngrok token from: https://dashboard.ngrok.com/get-started/your-authtoken

from pyngrok import ngrok, conf
import getpass

# Enter your ngrok auth token (get it from https://dashboard.ngrok.com/get-started/your-authtoken)
ngrok_token = getpass.getpass("Enter your ngrok auth token: ")
conf.get_default().auth_token = ngrok_token

# Create tunnel
public_url = ngrok.connect(11434, bind_tls=True)
print("\n" + "="*60)
print("🎉 Ollama is now accessible at:")
print(f"\n{public_url}")
print("\n" + "="*60)
print("\nUpdate your local .env file with:")
print(f"LLM_BASE_URL={public_url}")
print("\n" + "="*60)
print("\n⚠️  Keep this notebook running!")
print("The tunnel will close if you stop this cell or close the notebook.")

In [ ]:
# Keep the tunnel alive
# This cell will run indefinitely - DO NOT STOP IT
import time

print("Tunnel is active. Keep this cell running...")
print("Press the stop button to close the tunnel.\n")

try:
    while True:
        time.sleep(60)
        print(".", end="", flush=True)
except KeyboardInterrupt:
    print("\nTunnel closed.")